In [1]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = "svg"
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
from torchvision import models
import nltk
from nltk.corpus import stopwords
from collections import Counter
import string
import os

# Task 1

In [2]:
path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
df = pd.read_csv(f"{path}/train.csv")

Using Colab cache for faster access to the 'airline-passenger-satisfaction' dataset.


## Missing Values

In [3]:
df.drop(columns=["Unnamed: 0", "id", "Gender"], inplace=True)
df.dropna(inplace=True)

## Encoding

In [4]:
le = LabelEncoder()
df["Customer Type"] = le.fit_transform(df["Customer Type"])
df["Type of Travel"] = le.fit_transform(df["Type of Travel"])
df["satisfaction"] = le.fit_transform(df["satisfaction"])

In [5]:
ohe = OneHotEncoder(sparse_output=False)
class_encoded = ohe.fit_transform(df[["Class"]])
feature_names = ohe.get_feature_names_out(["Class"])
class_df = pd.DataFrame(class_encoded, columns=feature_names, index=df.index)
new_feature_names = ["Business", "Eco", "Eco Plus"]
class_df.columns = new_feature_names
df = pd.concat([df.drop("Class", axis=1), class_df], axis=1)

## Feature Scaling

In [6]:
scaler = StandardScaler()
cols_to_scale = df.columns.drop("satisfaction")

df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

## Review

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 103594 entries, 0 to 103903
Data columns (total 24 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Customer Type                      103594 non-null  float64
 1   Age                                103594 non-null  float64
 2   Type of Travel                     103594 non-null  float64
 3   Flight Distance                    103594 non-null  float64
 4   Inflight wifi service              103594 non-null  float64
 5   Departure/Arrival time convenient  103594 non-null  float64
 6   Ease of Online booking             103594 non-null  float64
 7   Gate location                      103594 non-null  float64
 8   Food and drink                     103594 non-null  float64
 9   Online boarding                    103594 non-null  float64
 10  Seat comfort                       103594 non-null  float64
 11  Inflight entertainment             103594 no

In [8]:
df.head()

,Customer Type,Age,Type of Travel,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,...,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction,Business,Eco,Eco Plus
0,-0.472883,-1.745542,1.491414,-0.731305,0.203521,0.616249,0.173716,-1.547312,1.352401,-0.185632,...,0.311853,0.549773,1.156211,1.305913,0.268966,0.072905,0,-0.957206,-0.904105,3.587718
1,2.114687,-0.951526,-0.670505,-0.956916,0.203521,-0.695032,0.173716,0.017981,-1.656487,-0.185632,...,-0.534854,-1.821038,0.305580,-1.742432,-0.360682,-0.237184,0,1.044708,-0.904105,-0.278729
2,-0.472883,-0.885358,-0.670505,-0.047454,-0.549571,-0.695032,-0.541118,-0.764666,1.352401,1.296479,...,0.311853,0.549773,0.305580,1.305913,-0.386917,-0.392229,1,1.044708,-0.904105,-0.278729
3,-0.472883,-0.951526,-0.670505,-0.629028,-0.549571,1.271890,1.603383,1.583273,-0.904265,-0.926688,...,-0.534854,-1.821038,0.305580,-0.980345,-0.098328,-0.159662,0,1.044708,-0.904105,-0.278729
4,-0.472883,1.430521,-0.670505,-0.977973,0.203521,-0.039391,0.173716,0.017981,0.600179,1.296479,...,0.311853,-0.240497,-0.545051,-0.218259,-0.386917,-0.392229,1,1.044708,-0.904105,-0.278729


In [9]:
df.shape

(103594, 24)

## Fully Connected Neural Network
### Training and Testing Data

In [10]:
X = df.drop("satisfaction", axis=1)
y = df["satisfaction"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

### Data preparation

In [11]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 64

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

print(f"Number of batches in train_loader: {len(train_loader)}")

Number of batches in train_loader: 1214


### Architecture

In [12]:
class AirlinePassengerSatisfactionClassifier(nn.Module):
    def __init__(self, input_features):
        super(AirlinePassengerSatisfactionClassifier, self).__init__()

        self.layer1 = nn.Linear(in_features=input_features, out_features=64)

        self.layer2 = nn.Linear(in_features=64, out_features=32)

        self.output_layer = nn.Linear(in_features=32, out_features=1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()


    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)

        x = self.layer2(x)
        x = self.relu(x)

        x = self.output_layer(x)
        x = self.sigmoid(x)

        return x


input_dimension = X_train_tensor.shape[1]
model = AirlinePassengerSatisfactionClassifier(input_features=input_dimension)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

cuda


### Loss and Optimizer

In [14]:
criterion = nn.BCELoss()
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Training Loop

In [15]:
epochs = 40
model.train()

for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    if (epoch + 1) == 1 or (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/40], Loss: 0.2215
Epoch [10/40], Loss: 0.0947
Epoch [20/40], Loss: 0.0856
Epoch [30/40], Loss: 0.0809
Epoch [40/40], Loss: 0.0765


### Testing and Evaluation

In [16]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = (correct / total) * 100
print(f"{accuracy:.2f}%")

96.00%


# Task 2
## Convolutional Neural Network
### Data preparation

In [17]:
batch_size = 64

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.SVHN(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.SVHN(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

### Architecture

In [18]:
class SVHNNet(nn.Module):
    def __init__(self):
        super(SVHNNet, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.flatten_size = 64 * 4 * 4

        self.fc1 = nn.Linear(self.flatten_size, 512)
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(512, 10)


    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(-1, self.flatten_size)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x


model = SVHNNet().to(device)

### Loss and Optimizer

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Training Loop

In [20]:
epochs = 10
model.train()

for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/10], Loss: 0.8544
Epoch [2/10], Loss: 0.4032
Epoch [3/10], Loss: 0.3281
Epoch [4/10], Loss: 0.2857
Epoch [5/10], Loss: 0.2542
Epoch [6/10], Loss: 0.2240
Epoch [7/10], Loss: 0.2007
Epoch [8/10], Loss: 0.1771
Epoch [9/10], Loss: 0.1611
Epoch [10/10], Loss: 0.1428


### Testing and Evaluation

In [21]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = (correct / total) * 100
print(f"{accuracy:.2f}%")

91.30%


## ResNet

In [22]:
resnet = models.resnet18(weights="DEFAULT")

resnet.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)

resnet.maxpool = nn.Identity()

features = resnet.fc.in_features
resnet.fc = nn.Linear(features, 10)

resnet = resnet.to(device)

### Optimizer

In [23]:
optimizer = optim.Adam(resnet.parameters(), lr=learning_rate)

### Training Loop

In [24]:
epochs = 5
resnet.train()

for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/5], Loss: 0.3718
Epoch [2/5], Loss: 0.2195
Epoch [3/5], Loss: 0.1818
Epoch [4/5], Loss: 0.1492
Epoch [5/5], Loss: 0.1209


### Testing and Evaluation

In [25]:
resnet.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = resnet(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = (correct / total) * 100
print(f"{accuracy:.2f}%")

94.98%


# Task 3
## Recurrent Neural Network

In [26]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [27]:
path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")
df = pd.read_csv(f"{path}/bbc-news-data.csv", sep="\t")

Using Colab cache for faster access to the 'bbcnewsarchive' dataset.


In [28]:
categories = ["business", "tech"]
df = df[df["category"].isin(categories)]

### Data preparation

In [29]:
stop_words = stopwords.words("english")


def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    filtered_words = [w for w in words if w not in stop_words]

    return " ".join(filtered_words)


df["clean_content"] = df["content"].apply(preprocess_text)

all_words = []
for text in df["clean_content"]:
    all_words.extend(text.split())

vocab_count = Counter(all_words)
most_common = vocab_count.most_common(5000)

vocab = {word: i+2 for i, (word, _) in enumerate(most_common)}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

max_len = 300


def text_to_indices(text):
    words = text.split()
    indices = [vocab.get(w, vocab["<UNK>"]) for w in words]

    if len(indices) < max_len:
        indices += [vocab["<PAD>"]] * (max_len - len(indices))
    else:
        indices = indices[:max_len]

    return indices


X_indices = df["clean_content"].apply(text_to_indices).tolist()
X_tensor = torch.tensor(X_indices, dtype=torch.long)

label_map = {"business": 0, "tech": 1}
y_indices = df["category"].map(label_map).tolist()
y_tensor = torch.tensor(y_indices, dtype=torch.float32).unsqueeze(1)

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor,
    y_tensor,
    random_state=0
)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 64

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

### Architecture

In [30]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(RNNClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size,
                                      embed_dimension,
                                      padding_idx=0)

        self.lstm = nn.LSTM(input_size=embed_dimension,
                            hidden_size=hidden_dimension,
                            batch_first=True)

        self.fc = nn.Linear(hidden_dimension, 1)
        self.sigmoid = nn.Sigmoid()


    def forward(self, x):
        x = self.embedding(x)
        output, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]
        x = self.fc(last_hidden)
        x = self.sigmoid(x)

        return x


vocab_size = len(vocab) + 1
embed_dimension = 100
hidden_dimension = 64

model = RNNClassifier(vocab_size, embed_dimension, hidden_dimension).to(device)

### Loss and Optimizer

In [31]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Training Loop

In [32]:
epochs=100
model.train()

for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    if (epoch + 1) == 1 or (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/100], Loss: 0.6882
Epoch [20/100], Loss: 0.5765
Epoch [40/100], Loss: 0.5042
Epoch [60/100], Loss: 0.4909
Epoch [80/100], Loss: 0.1270
Epoch [100/100], Loss: 0.1020


### Testing and Evaluation

In [33]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = (correct / total) * 100
print(f"{accuracy:.2f}%")

92.98%


## GloVe

In [34]:
if not os.path.exists("glove.6B.100d.txt"):
    !wget http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip
else:
    print("The GloVe file already exists")

--2025-12-11 19:47:28--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-12-11 19:47:28--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-12-11 19:47:28--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [35]:
embeddings_index = {}

with open("glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = coefs

embed_dimension = 100
vocab_size = len(vocab) + 1
embedding_matrix = np.zeros((vocab_size, embed_dimension))

hits = 0
misses = 0

for word, i in vocab.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
        hits += 1
    else:
        embedding_matrix[i] = np.random.normal(scale=0.6, size=(embed_dimension))
        misses += 1

print(f"Words found in GloVe: {hits}")
print(f"No words found: {misses}")

Words found in GloVe: 4881
No words found: 121


In [36]:
model = RNNClassifier(vocab_size, embed_dimension, hidden_dimension).to(device)

model.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

tensor([[-0.7328,  0.2403,  1.0150,  ..., -0.4299,  0.3519,  0.8728],
        [ 0.2334,  0.8432,  1.0013,  ...,  0.5909, -0.6904, -0.2078],
        [-0.1313, -0.4520,  0.0434,  ..., -0.3053, -0.0455,  0.5651],
        ...,
        [-0.1602, -1.0471,  0.5212,  ..., -0.1260,  0.2931,  0.3667],
        [ 0.5554, -0.4866, -0.3064,  ..., -1.4409, -0.4490, -0.2039],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       device='cuda:0')

### Optimizer

In [37]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Training Loop

In [38]:
epochs=100
model.train()

for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    if (epoch + 1) == 1 or (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/100], Loss: 0.6570
Epoch [20/100], Loss: 0.1366
Epoch [40/100], Loss: 0.0902
Epoch [60/100], Loss: 0.1090
Epoch [80/100], Loss: 0.0261
Epoch [100/100], Loss: 0.0196


### Testing and Evaluation

In [39]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = (correct / total) * 100
print(f"{accuracy:.2f}%")

98.68%
